In [ ]:
# Script adapted from: https://github.com/Sr933/rcc/tree/main
# Used to write US ccRCC validation data and Chromophobe
import h5py
import os

# Replace with your file path
main_folder=r"data\GSE159115_RAW"
# If True: output chromophobe RCC file, US ccRCC if not
chromophobe = False

for folder in os.listdir(main_folder):
    if ".h5" in folder:
        folder_path=os.path.join(main_folder, folder)
        print(folder.split("_")[0])
        

GSM4819732


In [2]:
import pandas as pd
import h5py
from scipy.sparse import csc_matrix
import os
import numpy as np
def load_h5_data(file_path, annotation_file):
    # Load the HDF5 data
    with h5py.File(file_path, 'r') as h5_file:
        group = h5_file['GRCh38']

        # Load datasets from the HDF5 file
        barcodes = group['barcodes'][:].astype(str)  # Convert bytes to strings
        data = group['data'][:]
        gene_names = group['gene_names'][:].astype(str)  # Gene names
        indices = group['indices'][:]
        indptr = group['indptr'][:]
        shape = tuple(group['shape'][:])

    # Reconstruct the sparse matrix using CSC format
    sparse_matrix = csc_matrix((data, indices, indptr), shape=shape)

    # Convert the sparse matrix to a Pandas DataFrame
    df = pd.DataFrame.sparse.from_spmatrix(sparse_matrix)

    # Load the annotation data
    annotation_df = pd.read_csv(annotation_file, compression='gzip')

    # Extract the barcode part from the cell column
    annotation_df['barcode'] = annotation_df.apply(lambda row: row['cell'].replace(f"{row['sample']}_", ""), axis=1)

    # Create a dictionary to map barcodes to annotations
    annotation_dict = dict(zip(annotation_df['barcode'], annotation_df['anno']))
    print(len(annotation_dict))
    # Map the barcodes in HDF5 data to their annotations
    df.index = gene_names
    df.columns = [annotation_dict.get(b, "Unknown") for b in barcodes]   # Column labels
    return df

In [ ]:
# Replace with your file path
label_file=r"path_to\us_sample_alloc_ch.csv"
path_anno_ben = r"path_to\GSE159115_normal_anno.csv.gz"
path_anno_tum = r"path_to\GSE159115_ccRCC_anno.csv.gz"
path_anno_chromophobe = r"path_to\GSE159115_chRCC_anno.csv.gz"
data=[]
labels=[]
label_df=pd.read_csv(label_file)

path_to_tumour = path_anno_chromophobe if chromophobe else path_anno_tum

for folder in os.listdir(main_folder):
    if ".h5" in folder:
        print(folder)
        file_path=os.path.join(main_folder, folder)
        for _, row in label_df.iterrows():
            sample_id = row['sample']  # Column name to match, adjust if different
            alloc=row['class']
            # Check if the sample ID is in the folder name
            if sample_id in folder:
                # Append the relevant row to the data list
                labels.append(alloc)
                break  # Exit loop once a match is found for this fol
        annotation_path=path_anno_ben if alloc=="Benign" else path_to_tumour
        print(annotation_path)
        df=load_h5_data(file_path, annotation_path)
        data.append(df)
        

print(labels)

GSM4819732_SI_21561_filtered_gene_bc_matrices_h5.h5
C:\Users\james\scRNA\RawDatasets\UnitedStates\GSE159115_chRCC_anno.csv.gz
2580
['chRCC']


In [7]:
combined_df = pd.concat(data, axis=1)
# Count how many "Unknown" columns there are
num_unknowns = (combined_df.columns == "Unknown").sum()
print(f"Number of columns with Unknown annotation: {num_unknowns}")

cleaned_df = combined_df.loc[:, ~combined_df.columns.str.contains("Unknown")]

# Print the shape of the DataFrame after cleaning
print(cleaned_df.shape)

Number of columns with Unknown annotation: 273
(33694, 2580)


In [8]:
print(list(set(cleaned_df)))

['Macro', 'ua', 'Tumor', 'Endo', 'vSMC', 'Tcell']


In [ ]:
import loompy

# Convert to .loom format
matrix = cleaned_df.values.astype('float32')  # Expression matrix
gene_names = cleaned_df.index.tolist()  # Gene names
cell_names = cleaned_df.columns.tolist()  # Cell type annotations

# Create sample class labels that match the actual number of cells
sample_class_labels = []
for i, df in enumerate(data):
    # Get number of cells in this sample (excluding "Unknown" columns)
    num_cells = (~df.columns.str.contains("Unknown")).sum()
    # Repeat the label for each cell in this sample
    sample_class_labels.extend([labels[i]] * num_cells)

print(f"Number of sample labels: {len(sample_class_labels)}")
print(f"Number of cells in cleaned_df: {cleaned_df.shape[1]}")

# Prepare row attributes (genes) and column attributes (cells)
row_attrs = {
    "gene_names": np.array(gene_names)
}

col_attrs = {
    "CellType": np.array(cell_names),
    "SampleClass": np.array(sample_class_labels)
}

# Write to .loom file
if chromophobe:
    output_path = r"path_to\chromophobeRCC.loom"
else:
    output_path = r"path_to\output.loom"
loompy.create(output_path, matrix, row_attrs, col_attrs)

print(f"Loom file created: {output_path}")

Number of sample labels: 2580
Number of cells in cleaned_df: 2580
Loom file created: C:\Users\james\scRNA\RawDatasets\UnitedStates\chromophobeRCC.loom
